In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/driver_race_base.csv")
print(df.shape)
df.head()

(25121, 14)


,season,round,raceId,race_date,circuitId,circuit_name,driverId,driver_name,constructorId,constructor_name,grid,positionOrder,statusId,target_top10
0,1950,1,833,1950-05-13,9,Silverstone Circuit,579,Juan Fangio,51,Alfa Romeo,3,12,44,0
1,1950,1,833,1950-05-13,9,Silverstone Circuit,589,Louis Chiron,105,Maserati,11,18,8,0
2,1950,1,833,1950-05-13,9,Silverstone Circuit,619,Bob Gerard,151,ERA,13,6,13,1
3,1950,1,833,1950-05-13,9,Silverstone Circuit,627,Louis Rosier,154,Talbot-Lago,9,5,12,1
4,1950,1,833,1950-05-13,9,Silverstone Circuit,640,Toulo de Graffenried,105,Maserati,8,17,5,0


In [3]:
df["race_date"] = pd.to_datetime(df["race_date"], errors="coerce")

df = df.sort_values(
    ["season", "round", "race_date", "raceId", "driverId"]
).reset_index(drop=True)

df[["season", "round", "race_date", "raceId"]].head()

,season,round,race_date,raceId
0,1950,1,1950-05-13,833
1,1950,1,1950-05-13,833
2,1950,1,1950-05-13,833
3,1950,1,1950-05-13,833
4,1950,1,1950-05-13,833


In [4]:
df["driver_experience"] = df.groupby("driverId").cumcount()

df[["driver_name", "season", "round", "driver_experience"]].head(10)

,driver_name,season,round,driver_experience
0,Juan Fangio,1950,1,0
1,Louis Chiron,1950,1,0
2,Bob Gerard,1950,1,0
3,Louis Rosier,1950,1,0
4,Toulo de Graffenried,1950,1,0
5,Nino Farina,1950,1,0
6,Johnny Claes,1950,1,0
7,Peter Walker,1950,1,0
8,Tony Rolt,1950,1,0
9,Prince Bira,1950,1,0


In [5]:
N_FORM = 5

df["driver_recent_form"] = (
    df.groupby("driverId")["positionOrder"]
      .shift(1)
      .rolling(N_FORM, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

df[["driver_name", "season", "round", "positionOrder", "driver_recent_form"]].head(15)

,driver_name,season,round,positionOrder,driver_recent_form
0,Juan Fangio,1950,1,12,NaN
1,Louis Chiron,1950,1,18,NaN
2,Bob Gerard,1950,1,6,NaN
3,Louis Rosier,1950,1,5,NaN
4,Toulo de Graffenried,1950,1,17,NaN
5,Nino Farina,1950,1,1,NaN
6,Johnny Claes,1950,1,11,NaN
7,Peter Walker,1950,1,20,NaN
8,Tony Rolt,1950,1,20,NaN
9,Prince Bira,1950,1,14,NaN


In [6]:
N_TEAM = 10

df["constructor_strength"] = (
    df.groupby("constructorId")["positionOrder"]
      .shift(1)
      .rolling(N_TEAM, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

df[["constructor_name", "season", "round", "positionOrder", "constructor_strength"]].head(15)

,constructor_name,season,round,positionOrder,constructor_strength
0,Alfa Romeo,1950,1,12,NaN
1,Maserati,1950,1,18,NaN
2,ERA,1950,1,6,NaN
3,Talbot-Lago,1950,1,5,NaN
4,Maserati,1950,1,17,18.000000
5,Alfa Romeo,1950,1,1,15.000000
6,Talbot-Lago,1950,1,11,11.666667
7,ERA,1950,1,20,10.250000
8,ERA,1950,1,20,12.200000
9,Maserati,1950,1,14,13.000000


In [7]:
def era_bucket(year):
    if year < 1970: return "1950-1969"
    if year < 1990: return "1970-1989"
    if year < 2010: return "1990-2009"
    if year < 2022: return "2010-2021"
    return "2022+"

df["era"] = df["season"].apply(era_bucket)
df["era"].value_counts()


era
1990-2009    7528
1970-1989    7388
2010-2021    5032
1950-1969    3854
2022+        1319
Name: count, dtype: int64

In [8]:
df["grid_bucket"] = pd.cut(
    df["grid"],
    bins=[0, 3, 10, 20, 60],
    labels=["front_row_1_3", "top10_4_10", "mid_11_20", "back_21+"]
)

df["grid_bucket"].value_counts(dropna=False)

grid_bucket
mid_11_20        10862
top10_4_10        7915
front_row_1_3     3391
back_21+          2953
Name: count, dtype: int64

In [9]:
feature_df = df[[
    "season", "round", "raceId", "race_date",
    "driverId", "driver_name",
    "constructorId", "constructor_name",
    "circuitId", "circuit_name",
    "grid",
    "driver_experience",
    "driver_recent_form",
    "constructor_strength",
    "era",
    "grid_bucket",
    "target_top10"
]].copy()

print(feature_df.isna().mean().sort_values(ascending=False).head(10))
feature_df.head()

driver_recent_form      0.002468
constructor_strength    0.000159
season                  0.000000
circuit_name            0.000000
grid_bucket             0.000000
era                     0.000000
driver_experience       0.000000
grid                    0.000000
circuitId               0.000000
round                   0.000000
dtype: float64


,season,round,raceId,race_date,driverId,driver_name,constructorId,constructor_name,circuitId,circuit_name,grid,driver_experience,driver_recent_form,constructor_strength,era,grid_bucket,target_top10
0,1950,1,833,1950-05-13,579,Juan Fangio,51,Alfa Romeo,9,Silverstone Circuit,3,0,NaN,NaN,1950-1969,front_row_1_3,0
1,1950,1,833,1950-05-13,589,Louis Chiron,105,Maserati,9,Silverstone Circuit,11,0,NaN,NaN,1950-1969,mid_11_20,0
2,1950,1,833,1950-05-13,619,Bob Gerard,151,ERA,9,Silverstone Circuit,13,0,NaN,NaN,1950-1969,mid_11_20,1
3,1950,1,833,1950-05-13,627,Louis Rosier,154,Talbot-Lago,9,Silverstone Circuit,9,0,NaN,NaN,1950-1969,top10_4_10,1
4,1950,1,833,1950-05-13,640,Toulo de Graffenried,105,Maserati,9,Silverstone Circuit,8,0,NaN,18.0,1950-1969,top10_4_10,0


In [10]:
feature_df["driver_recent_form"] = feature_df["driver_recent_form"].fillna(feature_df["driver_recent_form"].mean())
feature_df["constructor_strength"] = feature_df["constructor_strength"].fillna(feature_df["constructor_strength"].mean())

feature_df["driver_experience"] = feature_df["driver_experience"].fillna(0)


feature_df["grid_bucket"] = feature_df["grid_bucket"].astype("object").fillna("unknown")

print(feature_df.isna().mean().sort_values(ascending=False).head(10))

season                  0.0
circuit_name            0.0
grid_bucket             0.0
era                     0.0
constructor_strength    0.0
driver_recent_form      0.0
driver_experience       0.0
grid                    0.0
circuitId               0.0
round                   0.0
dtype: float64


In [12]:
feature_df.to_csv("../data/processed/driver_race_features.csv", index=False)
print("Saved: ../../data/processed/driver_race_features.csv", feature_df.shape)

Saved: ../../data/processed/driver_race_features.csv (25121, 17)
